<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/working_capstone_proj_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sec_edgar_downloader



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.3 MB/s eta 0:00:00


In [2]:
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = \
    "./triple-mountain-483601-k3-3a823d61bdb7.json"

In [ ]:
!pip install google-genai
from pydantic import BaseModel
from typing import List
from google import genai




class QAPair(BaseModel):
    question: str
    answer: str
    reference: str
    difficulty: str


class QAPairs(BaseModel):
    qa_pairs: List[QAPair]

In [4]:
import random

def random_flag_percent(random_modulo):
    random_int = random.randint(1, 100)
    #for e.g random_modulo is 5 For 20%
    ret = random_int%(int(random_modulo))
    if(ret == 0):
      return True
    return False

def random_int_range(low, high):
    random_int = random.randint(low, high)
    return random_int

In [5]:


def create_data_flag(chunk_num, section, total_chunks, num_data_created):
    # Need to take 20% of the chunks at random
    # Keep 100% of top 20% and bottom 20%
    # if there are hundred chunks, we need 20 chunks. 20% of 20 is 4
    # 4 records from top and 4 records from bottom are a must. Remaining
    # 12 records out of 92 at random but if total - num_created is less
    #than the desired number of records, then
    #chunk num 87, num created 8, total 92, desired 12

    #if((total - chunk num) <= (desired - num_created))
         #create the record

    desired_chunks = int(total_chunks*(0.2))
    desired_top_bottom_chunks = int(desired_chunks*(0.2))
    if((chunk_num <= desired_top_bottom_chunks) or
      (chunk_num >= (total_chunks - desired_top_bottom_chunks))):
        return True

    ret =  random_flag_percent(5)
    if(ret == False):
      if((total_chunks - chunk_num) <= (desired_chunks - num_data_created)):
        return True
    return ret





In [ ]:
#TEST RANDOMNESS

chunk_numbers = 57
i = 1
data_created = 0
while i < chunk_numbers:
  if(create_data_flag(i,1, chunk_numbers,data_created )):
    data_created = data_created +1
  i = i+1
print(f"created: {data_created}")

created: 11


In [6]:
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied: AAPL.json
All .json files copied successfully!


In [ ]:
!pip install sec-api
!pip install langchain-text-splitters


import requests
import json
import csv
from sec_api import ExtractorApi
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime
from google.colab import files

total_number_of_records = 0
client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
)

#TEST SECTION
parsed_cutoff_date = datetime.strptime("20201231", "%Y%m%d").date()
#parsed_cutoff_date = datetime.strptime("20241231", "%Y%m%d").date()

#TEST SECTION
sections = {"1", "1A", "7", "8"}
#sections = {"1"}
headers = {
    "User-Agent": "triple-mountain-483601-k3@appspot.gserviceaccount.com"
}

extractor = ExtractorApi(userdata.get('sec_api_key'))
positive_records = {}
#TEST SECTION
#with open('nasdaq50_cik.csv', mode='r', encoding='utf-8') as file:
with open('nasdaq1_cik.csv', mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)

    for row in reader:
        print(row)
        ticker = row[0]
        company  = row[1]
        cik = row[2]
        url = "https://data.sec.gov/submissions/CIK" + cik +".json"

        data = requests.get(url, headers=headers).content
        print(data)
        json_object = json.loads(data)

        recent = json_object["filings"]["recent"]

        forms = recent["form"]
        accessions = recent["accessionNumber"]
        dates = recent["filingDate"]
        #Get info about recent filing dates and types of filings
        for form, accession, date in zip(
          forms,
          accessions,
          dates):
          #Only get filings from last 5 years
          date = date.replace("-","")
          parsed_date = datetime.strptime(date, "%Y%m%d").date()
          if ((form == "10-K") & (parsed_date > parsed_cutoff_date)):

            accession = accession.replace("-","")
            print(date, accession)
            filing_url = "https://www.sec.gov/Archives/edgar/data/" + cik[2:] + "/" + accession + "/" + ticker + "-" + date +".txt"
            for section in sections:
              print(filing_url, section)
              text = extractor.get_section(
                   filing_url,
                    section,
                    "text"
                    )
              metadata = "Reference Ticker-" + ticker + " CompanyName-" + company + " Date-" + date + " Section-" + section
              #TODO:REMOVE TEXT
              #file_text = open(f"section.txt", "a", encoding="utf-8")
              #file_text.write(f"{metadata}\n")
              #file_text.write(f"{text}")
              #chunk the section text
              text_splitter = RecursiveCharacterTextSplitter(
                              separators=[
                              "\n\n",
                              "\n",
                              ". ",
                              " ",
                              ""
                              ],
                              chunk_size=500,
                              chunk_overlap=100
                              )

              chunks = text_splitter.split_text(text)

              print("Number of chunks in " + ticker + " " + section + " " +str(len(chunks)))
              chunk_num = 1
              num_data_created = 0
              total_chunks = len(chunks)
              record_name = (f"{ticker}-{section}-{date[:4]}")
              if record_name not in positive_records:
                positive_records[record_name] = []
              #file = open(f"{ticker}-{section}-{date[:4]}.txt", "a", encoding="utf-8")
              for chunk in chunks:
                #if (section == "8" or create_data_flag(chunk_num, section, total_chunks , num_data_created)):
                if (create_data_flag(chunk_num, section, total_chunks , num_data_created)):

                  prompt = f"""You are a financial analyst creating a training dataset. Given the SEC filing paragraph below with reference:{chunk}
                            Generate:
                             1. Two to three questions answerable solely from this paragraph.
                             2. The exact answer span from the paragraph.
                             3. Difficulty: easy, medium, or hard.
                             use company name {company} and year {date[:4]} when generating question. Return JSON only.
                         """
                  response = client.models.generate_content(
                           model="gemini-2.5-flash",
                           contents=prompt,
                           config={
                              "response_mime_type": "application/json",
                              "response_schema": QAPairs.model_json_schema()
                          }
                        )
                  resp_json = response.model_dump_json()
                  resp_dict = json.loads(resp_json)
                  qa_pairs = resp_dict['parsed']['qa_pairs']
                  for qa_pair in qa_pairs:
                    qa_pair['label'] = "1"
                    qa_pair['metadata'] = metadata
                    #print(qa_pair)
                    #file.write(f"{qa_pair}" + "\n")
                    positive_records[record_name].append(qa_pair)
                    num_data_created = num_data_created + 1
                    total_number_of_records = total_number_of_records + 1
                #TEST SECTION
                #if num_data_created >= 10:
                #  break
                chunk_num = chunk_num+1

              #file.close()
            print(f"Number of records for section: {num_data_created}")
            print(f"Number of records so far: {total_number_of_records}")

        data_file_name = (f"{ticker}.json")
        with open(data_file_name, "a", encoding="utf-8") as file_2:
        #with open("output2.json", "w") as file_2:
          json.dump(positive_records, file_2, indent=4)
        file_2.close()

        print(f"Total Number Of Records: {total_number_of_records}")
        files.download(f"{ticker}.json")


In [ ]:
#Copy data to drive
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied: AAPL.json
All .json files copied successfully!


In [ ]:
#Read the data from files

#companies = ["AAPL","MSFT","AMZN","NVDA","META","GOOGL","TSLA","AVGO","COST",
 #            "NFLX","AMD","AMAT","ASML","CSCO","QCOM","INTC","INTU","CMCSA",
  #           "TMUS","TXN","ADBE","PANW","AMGN","SBUX","ISRG","MDLZ","GILD",
   #          "BKNG","REGN","VRTX","ADP","MELI","ADI","KLAC","CTAS","SNPS",
    #         "CDNS","MAR","ORLY","NXPI","CRWD","WDAY","CTSH","ROST","LRCX",
     #        "FAST","PAYX","MCHP","AEP"]
import json

negative_records = {}
negative_records["Negative_Records"] = []
# Open the file in read mode ('r')
with open("output5.json", "r") as file:
    data = json.load(file)

#companies = ["AAPL","MSFT","AMZN","NVDA","META"]
companies = ["AAPL","MSFT"]
#TEST SECTION
years = ["2025", "2024", "2023", "2022", "2021"]
#years = ["2025"]
sections = ["1", "1A", "7", "8"]
#sections = ["1"]
file_records = {}
#Create 25% each of negative records
num_of_each_neg_rec_type = total_number_of_records*(0.95)
i = 0
while i < num_of_each_neg_rec_type:
  #pick 2 companies at ramdom
  company_index_1 = random_int_range(0, (len(companies) - 1 ))
  company_index_2 = random_int_range(0, (len(companies) - 1 ))

  #pick 2 sections at random
  section_index_1 = random_int_range(0, (len(sections) - 1 ))
  section_index_2 = random_int_range(0, (len(sections) - 1 ))
  #pick 2 years at random
  year_index_1 = random_int_range(0, (len(years) - 1 ))
  year_index_2 = random_int_range(0, (len(years) - 1 ))

  #25%
  if(i <  num_of_each_neg_rec_type):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_1] + "-" + years[year_index_1]

  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*2)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*3)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_2]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*4)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*5)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_2]

  records_1 = data[rec_name_1]
  records_2 = data[rec_name_2]

  #pick a random record index from first and second file
  record_index_1 = random_int_range(0, (len(records_1) - 1 ))
  record_index_2 = random_int_range(0, (len(records_2) - 1))



  record_3 = records_1[record_index_1]
  print(records_1[record_index_1])
  record_3['answer'] = records_2[record_index_2]['answer']
  record_3['metadata'] = records_2[record_index_2]['metadata']
  record_3['label'] = "0"
  print(record_3)
  negative_records["Negative_Records"].append(record_3)
  i = i+1







#TODO: write negative record to a file
with open("negative_output2.json", "w") as file_neg:
    json.dump(negative_records, file_neg, indent=4)

print(f"Total Negative Records: {i}")

#pick an index at random based on the size of the data in both
#copy the records and exchange the answers and label 0
#write to file


#create_negative_data_same_section_diff_comapny_diff_year(companies, sections, years)

{'question': "In 2025, which laptop models are part of Apple Inc.'s Mac product line?", 'answer': 'MacBook Air &#174; and MacBook Pro &#174;', 'reference': 'Products iPhone iPhone &#174; is the Company&#8217;s line of smartphones based on its iOS operating system. The iPhone line includes iPhone 17 Pro, iPhone Air&#8482;, iPhone 17, iPhone 16 and iPhone 16e. Mac Mac &#174; is the Company&#8217;s line of personal computers based on its macOS &#174; operating system. The Mac line includes laptops MacBook Air &#174; and MacBook Pro &#174; , as well as desktops iMac &#174; , Mac mini &#174; , Mac Studio &#174; and Mac Pro &#174; . iPad', 'difficulty': 'easy', 'label': '1', 'metadata': 'Reference Ticker-AAPL CompanyName-Apple Inc. Date-20251031 Section-1'}
{'question': "In 2025, which laptop models are part of Apple Inc.'s Mac product line?", 'answer': 'individuals and businesses', 'reference': 'Products iPhone iPhone &#174; is the Company&#8217;s line of smartphones based on its iOS operat

In [ ]:

import ast

def create_negative_data_same_section_diff_comapny_diff_year(companies,sections, years):

  #pick 2 companies at ramdom
  company_index_1 = int(0)
  company_index_2 = int(1)

  #pick 2 sections at random
  section_index_1 = int(0)
  section_index_2 = int(0)
  #pick 2 years at random
  year_index_1 = int(0)
  year_index_2 = int(1)

  file_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1] + ".txt"
  file_2 = companies[company_index_2]+ "-" + section[section_index_1] + "-" + years[year_index_1] + ".txt"


  records_1 = []

  with open(file_1, "r") as f1:
    for line in f1:
        records_1.append(ast.literal_eval(line.strip()))

  with open(file_2, "r") as f2:
    for line in f2:
        records_2.append(ast.literal_eval(line.strip()))

  #pick a random record index from first and second file
  record_index_1 = random_int_range(0, len(records_1))
  record_index_2 = random_int_range(0, len(record_2))



  record_3 = record_1[record_index_1]
  record_3['answer'] = record_2[record_index_2][answer]
  record_3['label'] = "0"
  print(f"record_3: {record_3}")
  #TODO: write negative record to a file
  file_neg = open(f"negatives.txt", "a", encoding="utf-8")
  file_neg.write(f"record_3\n")
  file.close()



  #pick an index at random based on the size of the data in both
  #copy the records and exchange the answers and label 0
  #write to file


def create_negative_data_same_section_same_company_diff_year():
  return
  #pick a company at ramdom
  #pick a section at random
  #pick 2 years at random
  #pick an index at random based on the size of the data in both
  #copy the records and exchange the answers and label 0
  #write to file

def create_negative_data_diff_section_same_company_same_year():
  return

  #pick a company at ramdom
  #pick 2 sections at random
  #pick a year at random
  #Exchange the answers and label it 0
  #copy the records and exchange the answers and label 0
  #write to file


def create_negative_data_diff_section_same_company_diff_year():
  return
  # pick a company at ramdom
  #pick 2 sections at random
  #pick 2 years at random
  #Exchange the answers and label it 0
  #copy the records and exchange the answers and label 0
  #write to file


def create_negative_data_diff_section_diff_comapny_diff_year():
  return
  # pick 2 companies at ramdom
  #pick 2 sections at random
  #pick 2 years at random
  #Exchange the answers and label it 0
  #copy the records and exchange the answers and label 0
  #write to file




In [ ]:
#Client and prompt for gemini vertex AI api to generate question and answer pairs
client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
)
tickers = {"AAPL","NVDA","MSFT"}
date_sections = {"1-2025", "1A-2025", "7-2025", "8-2025",
                  "1-2024", "1A-2024", "7-2024", "8-2024"
                  "1-2023", "1A-2023", "7-2023", "8-2023"
                  "1-2022", "1A-2022", "7-2022", "8-2022"
                  "1-2021", "1A-2021", "7-2021", "8-2021"                 }
for ticker in tickers:
  for date_section in date_sections:

    with open(f"Chunk-{ticker}-{date_section}.txt", "r") as chunk_file:
      for line in chunk_file:

        prompt = f"""You are a financial analyst creating a training dataset. Given the SEC filing paragraph below:{line}
                             Generate:
                             1. Two to three questions answerable solely from this paragraph.
                             2. The exact answer span from the paragraph.
                             3. Difficulty: easy, medium, or hard.
                             Return JSON only.
                          """
        response = client.models.generate_content(
                           model="gemini-2.5-flash",
                           contents=prompt,
                           config={
                              "response_mime_type": "application/json",
                              "response_schema": QAPairs.model_json_schema()
                           }
                          )
        resp_json = response.model_dump_json()
        resp_dict = json.loads(resp_json)
        qa_pairs = resp_dict['parsed']['qa_pairs']
        for qa_pair in qa_pairs:
          qa_pair['Metadata'] = metadata + " ChunkID-" + str(chunk_num)
          chunk_num = chunk_num+1
          #print(qa_pair)
          ile.write(f"{qa_pair}\n")

  file.close()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install sec-api
from sec_api import ExtractorApi


extractor = Extractor_Api("321cc89b2cf6aad0df9176d12ebab01d724ec756e173a834576bc307d54d299f")
filing_url = "https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20251231.htm"
text = extractor.get_section(
    filing_url,
    "7A",
    "text"
)

print(text[:1000])

In [ ]:
import ast

records = []


print(records[0]['answer'])

In [ ]:
# Declaring a simple dictionary
my_dictionary = {
    "name": "Alice",
    "age": 30,
    "city": "New York"
}

# Displaying the dictionary
print(my_dictionary)

# Accessing a value in the dictionary
print(f"Name: {my_dictionary['name']}")